# 1. Initialize Pipeline

## 1.1. Import Libraries

In [1]:
# (1) Import operating system functionalitie
import os
import shutil
import warnings
warnings.filterwarnings('ignore')               # Suppress all warnings

# (2) Import TensorFlow core libraries
import tensorflow as tf

# (3) Import TensorFlow Model Analysis for model evaluation
import tensorflow_model_analysis as tfma
from tensorflow_model_analysis.proto import config_pb2

# (4) Import TFX components for building an ML pipeline
from tfx.components import CsvExampleGen        # For ingesting CSV data
from tfx.components import StatisticsGen        # For generating statistics on your dataset

from tfx.components import SchemaGen            # For inferring the schema of your dataset
from tfx.v1.components import ImportSchemaGen   # For importing a predefined schema

from tfx.components import ExampleValidator     # For validating data examples against schema
from tensorflow_data_validation.utils.display_util import display_anomalies
from tensorflow_metadata.proto.v0 import anomalies_pb2

from tfx.components import Transform            # For feature engineering
from tfx.components import Trainer              # For training models
from tfx.components import Evaluator            # For evaluating model performance
from tfx.components import Pusher               # For deploying models to production

# (5) Import Resolver - Helps locate artifacts in the pipeline
from tfx.dsl.components.common.resolver import Resolver

# (6) Import latest_blessed_model_resolver - Find the latest validated model
from tfx.dsl.experimental import latest_blessed_model_resolver

# (7) Import protobuf message definitions for configuring components
from tfx.proto import example_gen_pb2           # For configuring example generation
from tfx.proto import trainer_pb2               # For configuring model training
from tfx.proto import pusher_pb2                # For configuring model deployment

# (8) Import Channel to define data connections between components
from tfx.types import Channel

# (9) Import standard artifact types for ML models
from tfx.types.standard_artifacts import Model, ModelBlessing

# (10) Import InteractiveContext to run the pipeline in an interactive environment like Jupyter notebooks
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

2026-01-13 11:10:24.113814: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-13 11:10:24.400766: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-13 11:10:24.400808: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-13 11:10:24.454430: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-13 11:10:24.566512: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-13 11:10:24.568495: I tensorflow/core/platform/cpu_feature_guard.cc:1

## 1.2. Set Data Directory

In [2]:
# Set the base, data, pipeline root, schema, and serving model directories
# TODO: replace with the ABSOLUTE path to your own working folder (e.g. /home/<your-username>/ff_ppln/exp).
#       DATA_DIR, PIPELINE_ROOT and SERVING_MODEL_DIR are all built from it.
BASE_DIR = "/path/to/your/ff_ppln/exp"

DATA_DIR = os.path.join(BASE_DIR, "exp_data")
PIPELINE_ROOT = os.path.join(BASE_DIR, "pl_root")
SCHEMA_DIR = os.path.join(PIPELINE_ROOT, "schema", "schema.pbtxt")
SERVING_MODEL_DIR = os.path.join(BASE_DIR, "model_serving")

# Set up context
context = InteractiveContext(pipeline_root=PIPELINE_ROOT)

# 2. ExampleGen Component

In [6]:
# Define the input configuration of the pre-existing splits
input = example_gen_pb2.Input(splits=[
                                        example_gen_pb2.Input.Split(name='train', pattern='span-{SPAN}/train/*'),
                                        example_gen_pb2.Input.Split(name='eval', pattern='span-{SPAN}/eval/*'),
                                        example_gen_pb2.Input.Split(name='test', pattern='span-{SPAN}/test/*')
                                    ])

# Create the CsvExampleGen component
example_gen = CsvExampleGen(input_base=DATA_DIR, input_config=input)

# Run the component
context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

# 3. StatisticsGen Component

In [7]:
# Create a StatisticsGen Component
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])

In [8]:
# Run the component
context.run(statistics_gen)

ExecutionResult(
    component_id: StatisticsGen
    execution_id: 2
    outputs:
        statistics: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=StatisticsGen, output_key=statistics, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [9]:
# Show the component
context.show(statistics_gen.outputs['statistics'])

# 4. SchemaGen Component

In [13]:
# Conditionally generate or import the scheme
if not os.path.exists(SCHEMA_DIR):
    
    print("Generating the schema...")
    # First run: generate the schema using the StatisticsGen output
    schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'], infer_feature_shape=True, exclude_splits=['eval', 'test'])
    
    # Run the schema
    context.run(schema_gen)
    
    # Save the schema to SCHEMA_DIR - When using Apache Beam, this code block shall go at the end (i.e., after running the pipeline).
    os.makedirs(os.path.dirname(SCHEMA_DIR), exist_ok=True)
    schema_artifact = schema_gen.outputs['schema'].get()[0]
    # Path to the schema file generated by SchemaGen
    generated_schema_path = os.path.join(schema_artifact.uri, "schema.pbtxt")
    # Copy the schema file to the desired location
    shutil.copy(generated_schema_path, SCHEMA_DIR)
    print(f"Schema generated and saved to {SCHEMA_DIR}")
    
else:
    
    print("Loading the schema...")
    
    # Import the Schema
    schema_gen = ImportSchemaGen(schema_file=SCHEMA_DIR)

    # Run the component
    context.run(schema_gen)
    
    print(f"Schema loaded successfully from {SCHEMA_DIR}")

Generating the schema...
Schema generated and saved to /path/to/your/ff_ppln/exp/pl_root/schema/schema.pbtxt


# 5. ExampleValidator Component

In [14]:
# Define ExampleValidator component
example_validator = ExampleValidator(
                                          statistics=statistics_gen.outputs['statistics'],
                                          schema=schema_gen.outputs['schema']
                                    )

# Run the component
context.run(example_validator)

ExecutionResult(
    component_id: ExampleValidator
    execution_id: 7
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [16]:
# Load the anomalies artifacts
anomalies_artifact = example_validator.outputs['anomalies'].get()[0]
anomalies_uri = anomalies_artifact.uri

# Define the split names
split_names = ["Split-train", "Split-eval", "Split-test"]

# Check for anomlies in each split
for split in split_names:
    print(f"Checking for anomalies - {split}")
    
    # Check for anomalies in all the splits
    anomalies_file_path = f"{anomalies_uri}/{split}/SchemaDiff.pb"

    # Load the Anomalies protocol buffer from the file
    with open(anomalies_file_path, 'rb') as f:
        anomalies_proto = anomalies_pb2.Anomalies()
        anomalies_proto.ParseFromString(f.read())

    # Check if there are any anomalies
    if anomalies_proto.anomaly_info:
        print("Anomalies detected in the dataset:")
        for feature_name, anomaly_info in anomalies_proto.anomaly_info.items():
            print(f"Feature: {feature_name}")
            print(f"Description: {anomaly_info.description}")
            print(f"Severity: {anomaly_info.severity}")
            print("------")
        
        # Stop the pipeline if anomalies are critical
        if any(anomaly_info.severity == anomalies_pb2.AnomalyInfo.Severity.ERROR for anomaly_info in anomalies_proto.anomaly_info.values()):
            print("Critical anomalies detected. Stopping the pipeline.")
            raise ValueError("Critical anomalies detected in the dataset. Pipeline stopped.")
            # Potentially send an email or notification to the team
    else:
        print("No anomalies detected in the dataset.")
        # Potentially send an email or notification to the team


Checking for anomalies - Split-train
No anomalies detected in the dataset.
Checking for anomalies - Split-eval
No anomalies detected in the dataset.
Checking for anomalies - Split-test
No anomalies detected in the dataset.


# 6. Transform Component

In [17]:
# Define a Transform component
transform = Transform(
                        examples=example_gen.outputs['examples'],
                        schema = schema_gen.outputs['schema'],
                        module_file=os.path.abspath('module.py')
                    )

# Run the component
context.run(transform)

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying module.py -> build/lib
installing to /tmp/tmpgyavvg50
running install
running install_lib
copying build/lib/module.py -> /tmp/tmpgyavvg50
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/tmpgyavvg50/tfx_user_code_Transform-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784395ea7d84a8ee8c897f6-py3.10.egg-info
running install_scripts
creating /tmp/tmpgyavvg50/tfx_user_code_Transform-0.0+85c11b8fb0003a50343182c3

/path/to/your/venv/lib/python3.10/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(


Processing ./pl_root/_wheels/tfx_user_code_Transform-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784395ea7d84a8ee8c897f6-py3-none-any.whl



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


Processing ./pl_root/_wheels/tfx_user_code_Transform-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784395ea7d84a8ee8c897f6-py3-none-any.whl


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 8
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

# 7. Tuner Component

In [19]:
from tfx.components import Tuner
from tfx.proto import trainer_pb2

tuner = Tuner(
                module_file=os.path.abspath('module.py'),
                examples=transform.outputs['transformed_examples'],
                transform_graph=transform.outputs['transform_graph'],
                train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=81),
                eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=41)
            )

context.run(tuner)

Trial 3 Complete [00h 00m 07s]
val_loss: 38.87937927246094

Best val_loss So Far: 38.87937927246094
Total elapsed time: 00h 00m 25s
Results summary
Results in /path/to/your/ff_ppln/exp/pl_root/.temp/10/dnn_bayes
Showing 10 best trials
Objective(name="val_loss", direction="min")

Trial 2 summary
Hyperparameters:
num_hidden_layers: 2
dense_units_0: 64
activation_0: tanh
dropout_0: 0.30000000000000004
dense_units_1: 192
activation_1: relu
dropout_1: 0.2
dense_units_2: 128
activation_2: relu
dropout_2: 0.30000000000000004
Score: 38.87937927246094

Trial 1 summary
Hyperparameters:
num_hidden_layers: 3
dense_units_0: 96
activation_0: tanh
dropout_0: 0.4
dense_units_1: 32
activation_1: relu
dropout_1: 0.1
dense_units_2: 32
activation_2: relu
dropout_2: 0.1
Score: 39.12489318847656

Trial 0 summary
Hyperparameters:
num_hidden_layers: 1
dense_units_0: 64
activation_0: relu
dropout_0: 0.5
Score: 39.66078567504883


ExecutionResult(
    component_id: Tuner
    execution_id: 10
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

## 7.1. Best Hyperparameters

In [20]:
import os
import json
from keras_tuner.engine.hyperparameters import HyperParameters

# Step 1: Get URI
hparams_uri = tuner.outputs['best_hyperparameters'].get()[0].uri

# Step 2: Read the JSON file
with open(os.path.join(hparams_uri, 'best_hyperparameters.txt')) as f:
    hp_config = json.load(f)

# Step 3: Load the HyperParameters object
hp = HyperParameters.from_config(hp_config)

# Step 4: View the best values
print(hp.values)

{'num_hidden_layers': 2, 'dense_units_0': 64, 'activation_0': 'tanh', 'dropout_0': 0.30000000000000004, 'dense_units_1': 192, 'activation_1': 'relu', 'dropout_1': 0.2, 'dense_units_2': 128, 'activation_2': 'relu', 'dropout_2': 0.30000000000000004}


# 8. Trainer Component

In [22]:
# Define a Trainer component
trainer = Trainer(
                    module_file=os.path.abspath('module.py'),
                    transformed_examples=transform.outputs['transformed_examples'],
                    schema=schema_gen.outputs['schema'],
                    transform_graph=transform.outputs['transform_graph'],
                    hyperparameters=tuner.outputs['best_hyperparameters'],
                    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=81),
                    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=41)
                )

# Run the component
context.run(trainer)

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying module.py -> build/lib
installing to /tmp/tmp6djfwq06
running install
running install_lib
copying build/lib/module.py -> /tmp/tmp6djfwq06
running install_egg_info
running egg_info
creating tfx_user_code_Trainer.egg-info
writing tfx_user_code_Trainer.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Trainer.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Trainer.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Trainer.egg-info/SOURCES.txt'
Copying tfx_user_code_Trainer.egg-info to /tmp/tmp6djfwq06/tfx_user_code_Trainer-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784395ea7d84a8ee8c897f6-py3.10.egg-info
running install_scripts
creating /tmp/tmp6djfwq06/tfx_user_code_Trainer-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784

/path/to/your/venv/lib/python3.10/site-packages/setuptools/command/install.py:34: SetuptoolsDeprecationWarning: setup.py install is deprecated. Use build and pip and other standards-based tools.
  warnings.warn(


Processing ./pl_root/_wheels/tfx_user_code_Trainer-0.0+85c11b8fb0003a50343182c3241b5abd1dc2748ce784395ea7d84a8ee8c897f6-py3-none-any.whl



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


Epoch 1/70
81/81 [==============================] - 1s 5ms/step - loss: 1354.3358 - mae: 34.8472 - rmse: 36.8013 - val_loss: 1292.8083 - val_mae: 34.0046 - val_rmse: 35.9556 - lr: 0.0010
Epoch 2/70
81/81 [==============================] - 0s 3ms/step - loss: 1082.4525 - mae: 31.6698 - rmse: 32.9006 - val_loss: 876.2575 - val_mae: 27.9552 - val_rmse: 29.6016 - lr: 0.0010
Epoch 3/70
81/81 [==============================] - 0s 3ms/step - loss: 716.4895 - mae: 25.4188 - rmse: 26.7673 - val_loss: 415.2379 - val_mae: 18.5870 - val_rmse: 20.3774 - lr: 0.0010
Epoch 4/70
81/81 [==============================] - 0s 3ms/step - loss: 338.6209 - mae: 16.5378 - rmse: 18.4017 - val_loss: 133.2994 - val_mae: 9.4687 - val_rmse: 11.5455 - lr: 0.0010
Epoch 5/70
81/81 [==============================] - 0s 3ms/step - loss: 117.7454 - mae: 8.9455 - rmse: 10.8511 - val_loss: 52.7792 - val_mae: 5.3258 - val_rmse: 7.2649 - lr: 0.0010
Epoch 6/70
81/81 [==============================] - 0s 3ms/step - loss: 71.79

...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: /path/to/your/ff_ppln/exp/pl_root/Trainer/model/12/Format-Serving/assets


INFO:tensorflow:Assets written to: /path/to/your/ff_ppln/exp/pl_root/Trainer/model/12/Format-Serving/assets


ExecutionResult(
    component_id: Trainer
    execution_id: 12
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

# 9. Model Evaluator Component

## 9.1. Latest Blessed Model

In [23]:
# 1. Get the latest blessed model
model_resolver = Resolver(
                              strategy_class = latest_blessed_model_resolver.LatestBlessedModelResolver,
                              model = Channel(type=Model),
                              model_blessing = Channel(type=ModelBlessing)
                              ).with_id('latest_blessed_model_resolver')

# Run the component
context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 13
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [24]:
latest_model = model_resolver.outputs['model'].get()

if latest_model:
    print("Latest Blessed Model URI:", latest_model[0].uri)
else:
    print("⚠️ No blessed model found in MLMD (first run or no model was blessed).")

⚠️ No blessed model found in MLMD (first run or no model was blessed).


### 9.2. Evaluator Component

Running the below component, without any change in the pipeline (i.e., model or data) will result in an error, because this component will be attempting to recreate an artifact that is already existing in the Metadata Store--which violates the Metadata Store constraints, leading to an error!

In [25]:
# Define evaluation configuration for a single model that handles multiple outputs
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(
            signature_name='serving_default',
            label_key='c_str',
            prediction_key="predictions"
        )
    ],
    slicing_specs=[],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                        tfma.MetricConfig(
                            class_name="MeanAbsoluteError",
                            threshold=tfma.MetricThreshold(
                                # Threshold for blessing: MAE < 10.0
                                value_threshold=tfma.GenericValueThreshold(
                                    upper_bound={'value': 10.0}
                                ),
                                # For subsequent runs, ensure new model is better than baseline
                                change_threshold=tfma.GenericChangeThreshold(
                                    direction=tfma.MetricDirection.LOWER_IS_BETTER,
                                    absolute={'value': -0.1}  # New model must improve MAE by at least 0.1
                                )
                            )
                        ),

                        tfma.MetricConfig(
                            class_name="RootMeanSquaredError",
                            threshold=tfma.MetricThreshold(
                                # Threshold for blessing: RMSE < 15.0
                                value_threshold=tfma.GenericValueThreshold(
                                    upper_bound={'value': 15.0}
                                ),
                                # For subsequent runs, ensure new model is better than baseline
                                change_threshold=tfma.GenericChangeThreshold(
                                    direction=tfma.MetricDirection.LOWER_IS_BETTER,
                                    absolute={'value': -0.1}  # New model must improve RMSE by at least 0.1
                                )
                            )
                        )
            ]
        )
    ]
)

# Define Evaluator component
evaluator = Evaluator(
                        examples=example_gen.outputs["examples"],
                        model=trainer.outputs["model"],
                        baseline_model=model_resolver.outputs["model"],
                        eval_config=eval_config,
                        example_splits=['test']
)

# Run the component
context.run(evaluator)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 14
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [26]:
blessing = evaluator.outputs['blessing'].get()
print("Blessed:", blessing[0].mlmd_artifact.custom_properties['blessed'].int_value)

Blessed: 1


# 10. Pusher Component

In [27]:
# Define Pusher component
pusher = Pusher(
    model = trainer.outputs["model"],
    model_blessing = evaluator.outputs["blessing"],
    push_destination = pusher_pb2.PushDestination(
        filesystem = pusher_pb2.PushDestination.Filesystem(base_directory = SERVING_MODEL_DIR)
    ),
    custom_config={'transform_graph': transform.outputs['transform_graph']} # pass the transform graph to the pusher
)

# Run the component
context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 15
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False))

In [28]:
# Show the outputs
pusher.outputs

{'pushed_model': OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}, _input_trigger=None, _is_async=False)}

In [29]:
# Get the URI of the pushed model from the TFX Pusher component output
push_uri = pusher.outputs['pushed_model'].get()[0].uri

# Load the SavedModel from the retrieved URI
model = tf.saved_model.load(push_uri)

# Iterate through all signature items in the model
for item in model.signatures.items():
  print(item)

('serving_default', <ConcreteFunction (*, examples: TensorSpec(shape=(None,), dtype=tf.string, name='examples')) -> Dict[['predictions', TensorSpec(shape=(None,), dtype=tf.float32, name='predictions')]] at 0x7F1084105AB0>)
('serve_json', <ConcreteFunction (*, age: TensorSpec(shape=(None,), dtype=tf.int64, name='age'), bfs: TensorSpec(shape=(None,), dtype=tf.float32, name='bfs'), ca: TensorSpec(shape=(None,), dtype=tf.float32, name='ca'), cement: TensorSpec(shape=(None,), dtype=tf.float32, name='cement'), fa: TensorSpec(shape=(None,), dtype=tf.float32, name='fa'), fa_1: TensorSpec(shape=(None,), dtype=tf.float32, name='fa_1'), sp: TensorSpec(shape=(None,), dtype=tf.float32, name='sp'), water: TensorSpec(shape=(None,), dtype=tf.float32, name='water')) -> Dict[['predictions', TensorSpec(shape=(None,), dtype=tf.float32, name='predictions')]] at 0x7F10F02FB1C0>)


In [30]:
predict_json = model.signatures["serve_json"]

In [31]:
example = {
    "cement": tf.constant([540.0], dtype=tf.float32),
    "bfs":    tf.constant([0.0],   dtype=tf.float32),
    "fa":     tf.constant([0.0],   dtype=tf.float32),
    "water":  tf.constant([162.0], dtype=tf.float32),
    "sp":     tf.constant([2.5],   dtype=tf.float32),
    "ca":     tf.constant([1040.0],dtype=tf.float32),
    "fa_1":   tf.constant([676.0], dtype=tf.float32),  # note: underscore, not dot
    "age":    tf.constant([28],    dtype=tf.int64),
}

out = predict_json(**example)
print(out["predictions"].numpy())

[61.751675]


In [35]:
batch = {
    "cement": [540.0, 450.0],
    "bfs":    [0.0,   0.0],
    "fa":     [0.0,   0.0],
    "water":  [162.0, 170.0],
    "sp":     [2.5,   3.5],
    "ca":     [1040.0, 760.0],
    "fa_1":   [676.0, 600.0],
    "age":    [28,   21],
}

inputs = {k: tf.constant(v, dtype=tf.float32 if k != "age" else tf.int64) for k,v in batch.items()}
outB = predict_json(**inputs)
print(outB["predictions"].numpy())

[61.751682 23.079628]
